# Seizure detection sandbox

## Workspace Preparation

In [4]:
# iEEG imports
from ieeg.auth import Session

# Scientific computing imports
import numpy as np
import scipy as sc
import pandas as pd
import json
from scipy.linalg import hankel
from tqdm import tqdm
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import minmax_scale

# Data IO imports
import mne
import mne_bids
from mne_bids import BIDSPath, read_raw_bids

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Imports for deep learning
import random
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
import tensorflow as tf
from tensorflow.keras.models import load_model

# OS imports
import os
from os.path import join as ospj
from os.path import exists as ospe
from utils import *
import sys
sys.path.append('/users/wojemann/DynaSD')
from DynaSD import NDD,GIN,LiNDDA,MINDA
from config import Config

# Get paths from config
datapath = Config.datapath
prodatapath = Config.prodatapath
metapath = Config.metapath
figpath = Config.figpath

# Load data

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)  # Memory growth must be set before GPUs have been initialized


2025-09-02 14:56:36.472675: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-02 14:56:36.521560: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-02 14:56:36.524440: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [4]:
_,_,datapath,prodatapath,figpath,patient_table,rid_hup,_ = load_config(ospj('/mnt/leif/littlab/users/wojemann/stim-seizures/code','config.json'))

In [5]:
set_seed(5210)

In [64]:
pt = 'HUP249'
montage = 'bipolar'

In [65]:
all_seizure_times = pd.read_csv(ospj(prodatapath,"consensus_annots.csv"))
seizure_times = all_seizure_times[all_seizure_times.patient == pt]
# seizure_times.head()

## Anomaly Detection

### Generating model predictions

In [66]:
from seizure_detection_pipeline import *

In [67]:
seizures_df = pd.read_csv(ospj(datapath,"stim_seizure_information_BIDS.csv"))

In [68]:
def preprocess_for_detection(data,fs,montage='bipolar',target=256, wavenet=False, pre_mask = None):
    # This function implements preprocessing steps for seizure detection
    chs = data.columns.to_list()
    ch_df = check_channel_types(chs)
    # Montage
    if montage == 'bipolar':
        data_bp_np,bp_ch_df = bipolar_montage(data.to_numpy().T,ch_df)
        bp_ch = bp_ch_df.name.to_numpy()
    elif montage == 'car':
        data_bp_np = (data.to_numpy().T - np.mean(data.to_numpy(),1))
        bp_ch = chs
    
    # Channel rejection
    if pre_mask is None:
        mask,_ = detect_bad_channels(data_bp_np.T,fs)
        data_bp_np = data_bp_np[mask,:]
        bp_ch = bp_ch[mask]
    else:
        data_bp_np = data_bp_np[pre_mask,:]
        bp_ch = bp_ch[pre_mask]
    
    if wavenet:
        target=128
        data_bp_notch = notch_filter(data_bp_np,fs)
        data_bp_filt = bandpass_filter(data_bp_notch,fs,lo=3,hi=127)
        signal_len = int(data_bp_filt.shape[1]/fs*target)
        data_bpd = sc.signal.resample(data_bp_filt,signal_len,axis=1).T
        fsd = int(target)
    else:
        # Bandpass filtering
        # b,a = sc.signal.butter(4,[3,40],btype='bandpass',fs = fs)
        # data_bp_filt = sc.signal.filtfilt(b,a,data_bp_np,axis=1)
        data_bp_notch = notch_filter(data_bp_np,fs)
        data_bp_filt = bandpass_filter(data_bp_notch,fs,lo=3,hi=100)
        # Down sampling
        signal_len = int(data_bp_filt.shape[1]/fs*target)
        data_bpd = sc.signal.resample(data_bp_filt,signal_len,axis=1).T
        fsd = int(target)
    data_white = ar_one(data_bpd)
    data_white_df = pd.DataFrame(data_white,columns = bp_ch)
    if pre_mask is None:
        return data_white_df,fsd,mask
    else:
        return data_white_df,fsd

In [69]:
# Loading data from bids
inter,fs_raw = get_data_from_bids(ospj(datapath,"BIDS"),pt,'interictal')
# Pruning channels
chn_labels = remove_scalp_electrodes(inter.columns)
inter = inter[chn_labels]
try:
    electrode_localizations,electrode_regions = electrode_wrapper(pt,rid_hup,datapath)
    electrode_localizations.name = clean_labels(electrode_localizations.name,pt)
    electrode_regions.name = clean_labels(electrode_regions.name,pt)
    electrode_localizations.to_pickle(ospj(prodatapath,pt,'electrode_localizations_atropos.pkl'))
    electrode_regions.to_pickle(ospj(prodatapath,pt,'electrode_localizations_dkt.pkl'))
    neural_channels = electrode_localizations.name[(electrode_localizations.name.isin(inter.columns)) & ((electrode_localizations.label == 'white matter') | (electrode_localizations.label == 'gray matter'))]
except:
    print(f"electrode localization failed for {pt}")
    neural_channels = chn_labels
inter_neural = inter.loc[:,neural_channels]

# Detecting and removing excess noisy channels
# mask,_ = detect_bad_channels(inter.to_numpy(),fs)
# inter = inter.drop(inter.columns[~mask],axis=1)
inter,fs,mask = preprocess_for_detection(inter_neural,fs_raw,wavenet=False,target=128)